In [27]:
from local.constants import WORKSPACE_ROOT
from pathlib import Path
import pandas as pd
import numpy as np

In [21]:
book = pd.ExcelFile(WORKSPACE_ROOT/"data/lipids/growth_kinetics/Growth_Kinetics_Toluene_5Jan2026.xlsx")
df = book.parse('Growth_Kinetics_Toluene_5Jan202', header=None)

entries = []
for _, r in df.iloc[9:, 1:15].iterrows():
    tol_ara = r[0:3]
    ara = r[3:6]
    tol = r[6:9]
    non = r[9:12]
    t = r[13][:-1] # hours
    clone, media = r[14].split("_")

    for i, (a, b, c, d) in enumerate(zip(tol_ara, ara, tol, non)):
        for v, cond in [
            (a, "tol+ ara+"),
            (b, "tol- ara+"),
            (c, "tol+ ara-"),
            (d, "tol- ara-"),
        ]:
            entries.append((clone, media, cond, t, i+1, v))
df = pd.DataFrame(entries, columns="clone, media, condition, time, replicate, OD".split(", "))
df

,clone,media,condition,time,replicate,OD
0,Stan,M9,tol+ ara+,0,1,0.219
1,Stan,M9,tol- ara+,0,1,0.207
2,Stan,M9,tol+ ara-,0,1,0.282
3,Stan,M9,tol- ara-,0,1,0.286
4,Stan,M9,tol+ ara+,0,2,0.200
...,...,...,...,...,...,...
3643,Epi300,LB,tol- ara-,148,2,0.456
3644,Epi300,LB,tol+ ara+,148,3,0.475
3645,Epi300,LB,tol- ara+,148,3,0.495
3646,Epi300,LB,tol+ ara-,148,3,0.453


In [22]:
from local.figures.template import BaseFigure, ApplyTemplate, go
from local.figures.colors import Color, COLORS, Palettes
from local.caching import save, load, save_exists

In [24]:
df.condition.unique()

array(['tol+ ara+', 'tol- ara+', 'tol+ ara-', 'tol- ara-'], dtype=object)

In [97]:
rows_map = {
    ("Stan",    "M9"): 1,
    ("Scot",    "M9"): 2,
    ("pCC1",    "M9"): 3,
    ("Epi300",  "M9"): 4,
    ("Stan",    "LB"): 5,
    ("Scot",    "LB"): 6,
    ("pCC1",    "LB"): 7,
    ("Epi300",  "LB"): 8,
}
cols_map = {
    "tol+ ara+": 1,
    "tol- ara+": 2,
    "tol+ ara-": 3,
    "tol- ara-": 4,
}

ROWS = 8
COLS = 4
fig = BaseFigure(
    shape=(COLS, ROWS),
    shared_yaxes=True, shared_xaxes=True,
    horizontal_spacing=0.01, vertical_spacing=0.01,
    y_title="OD",
    x_title="time (hours)",
    # subplot_titles=[_format_name(name) for name, _ in _todo]
)
for i, ann in enumerate(fig.layout.annotations):
    a, b = ann.x, ann.y
    if ann.x == 0.5: # x title
        ann.y -= 0.03
    else: # y title
        ann.x -= 0.045

panels = {}
for _, df_row in df.iterrows():
    row = rows_map[(df_row.clone, df_row.media)]
    col = cols_map[df_row.condition]
    k = (col, row)
    panels[k] = panels.get(k, [])+[(df_row.replicate, df_row.time, df_row.OD)]

for (col, row), vals in panels.items():
    mat = np.array(vals, dtype=float)
    for i in [1, 2, 3]:
        _f = mat[:, 0] == i
        rep = mat[_f, 1:]
        rep = rep[:9] # stop at 32 h
        fig.add_trace(
            go.Scatter(
                x = [x for x, y in rep],
                y = [y for x, y in rep],
                marker=dict(
                size=6,
                color=COLORS.TRANSPARENT,
                line=dict(
                    width=1,
                    color=Color.Hex("212121").color_value,
                )
            ),
            line=dict(
                width=1,
                color=Color.Hex("212121").Fade(0.5).color_value,
            ),
                showlegend=False,
            ),
            col=col, row=row,
        )
axis_hide = dict(ticks=None, showticklabels=False, showline=False)
axis_show = dict(ticks="outside", showticklabels=True, showline=True)
row_titles = list(rows_map)
col_titles = list(cols_map)
fig = ApplyTemplate(
    fig,
    default_xaxis=axis_hide|dict(showline=True),
    default_yaxis=axis_hide,
    axis = {
        f"1 {y+1} y": axis_show|dict(range=[0, 1.1], title=dict(text=" ".join(row_titles[y]), font_size=16))
        for y in range(ROWS)
    } | {
        f"{x+1} {ROWS} x": axis_show|dict(dtick=8, title=dict(text=col_titles[x], font_size=16))
        for x in range(COLS)
    },
    layout=dict(
        width=800, height=1200,
        margin=dict(l=105, r=15, t=45, b=105),
    )
)
fig.show()
fig.write_image("./cache/ScaDC_growth_kinetics.svg")